In [3]:
import pandas as pd
import numpy as np

In [163]:
np.set_printoptions(precision=2, suppress=True)

In [4]:
ratings = pd.read_csv(
    r"C:\Users\user\Downloads\ml-100k\ml-100k\u.data",
    sep="\t",
    names=["user_id" , "movie_id", "rating" , "timestamp"]
)

In [5]:
print(ratings.head(10))

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596
5      298       474       4  884182806
6      115       265       2  881171488
7      253       465       5  891628467
8      305       451       3  886324817
9        6        86       3  883603013


In [6]:
print(ratings.shape)

(100000, 4)


In [7]:
ratings["user_id"].nunique()

943

In [8]:
ratings["movie_id"].nunique()

1682

In [9]:
print(ratings["rating"].value_counts().sort_index())

rating
1     6110
2    11370
3    27145
4    34174
5    21201
Name: count, dtype: int64


In [10]:
rating_matrix = ratings.pivot(index="user_id",columns="movie_id",values="rating")

In [139]:
print(rating_matrix.shape)

(943, 1682)


In [142]:
film_num = 1682
user_num = 943

In [143]:
small_matrix=rating_matrix.copy()

In [144]:
print(small_matrix.shape)

(943, 1682)


In [145]:
print(small_matrix)

movie_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                               ...   
1          5.0   3.0   4.0   3.0   3.0   5.0   4.0   1.0   5.0   3.0  ...   
2          4.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   2.0  ...   
3          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
5          4.0   3.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
...        ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
939        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   5.0   NaN  ...   
940        NaN   NaN   NaN   2.0   NaN   NaN   4.0   5.0   3.0   NaN  ...   
941        5.0   NaN   NaN   NaN   NaN   NaN   4.0   NaN   NaN   NaN  ...   
942        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
943        NaN   5.0   NaN   NaN   NaN   NaN   NaN   NaN   3.0   NaN  ...   

In [183]:
yArray=np.array(small_matrix.copy())
nan_mask = np.isnan(yArray[:,:])
yArray[nan_mask] = 0
print(yArray)

[[5. 3. 4. ... 0. 0. 0.]
 [4. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [5. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 5. 0. ... 0. 0. 0.]]


In [147]:
rArray=np.array(small_matrix.copy())
rArray[nan_mask] = 0
rArray[~nan_mask] = 1
print(rArray)
print(rArray.shape)

[[1. 1. 1. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]]
(943, 1682)


In [152]:
def costFunc(W, X, B_film, B_user):
    error = W @ X.T + B_user[:,np.newaxis] + B_film[np.newaxis,:] - yArray
    cost = rArray * error
    cost = np.sum(np.square(cost)) / 2
    return cost

In [202]:
def predict(W, X, B_film, B_user):
    global_mean = np.sum(yArray) / np.sum(rArray)
    prediction_matrix = W @ X.T + B_user[:,np.newaxis] + B_film[np.newaxis,:] + global_mean
    return prediction_matrix

In [203]:
np.array([1,2,0.5])[:,np.newaxis]

array([[1. ],
       [2. ],
       [0.5]])

In [204]:
def gradientDescent(W, X, B_film, B_user, iteration, learning_rate, momentum = 0.8 ,lambda_ = 0.1):

    
    dW = None; dX = None; dB_film = None; dB_user = None;
    
    vW = np.zeros_like(W)
    vX = np.zeros_like(X)
    vB_film = np.zeros_like(B_film)
    vB_user = np.zeros_like(B_user)
    
    
    for i in range(iteration + 1):
        
        prediction = predict(W, X, B_film, B_user)
        error = (prediction - yArray) * rArray
        
        dW = np.dot(error, X) + lambda_ * W
        dX = np.dot(error.T, W) + lambda_ * X
        
        dB_film = np.sum(error, axis = 0)
        dB_user = np.sum(error, axis = 1)

        vW = momentum * vW + learning_rate * dW
        vX = momentum * vX + learning_rate * dX
        vB_film = momentum * vB_film + learning_rate * dB_film
        vB_user = momentum * vB_user + learning_rate * dB_user
        
        W = W - vW
        X = X - vX
        B_film = B_film - vB_film
        B_user = B_user - vB_user

        if i % 100 == 0:
            data_loss = np.sum( np.square(error) / 2)
            regularization_loss = (lambda_ / 2) * (np.sum(np.square(W)) + np.sum(np.square(X)))
            total_loss = data_loss + regularization_loss
            print(i,"th iteration     Loss:", "{:.4f}".format(total_loss))

    return W, X, B_film, B_user

In [205]:
def calculate_errors(W, X, B_film, B_user):
    n = np.sum(rArray)
    prediction = predict(W, X, B_film, B_user)
    error = (prediction - yArray) * rArray        
    
    mse = np.sum(np.square(error)) / n
    rmse = np.sqrt(mse)
    mae = np.sum(np.abs(error)) / n
    
    print("MSE:", mse)
    print("RMSE:", rmse)
    print("MAE:", mae)

In [206]:
feature_num = 40
W = np.zeros((user_num,feature_num))
X = np.zeros((film_num,feature_num))
B_film = np.zeros((film_num,))
B_user = np.zeros((user_num,))
W , X, B_film, B_user = gradientDescent(W, X, B_film, B_user, 500, 0.0015,0.001)

0 th iteration     Loss: 63356.4190
100 th iteration     Loss: 41905.9126
200 th iteration     Loss: 41791.3821
300 th iteration     Loss: 41754.8960
400 th iteration     Loss: 41736.8584
500 th iteration     Loss: 41726.2161


In [207]:
calculate_errors(W, X, B_film, B_user)

MSE: 0.8345226474687223
RMSE: 0.9135221111000665
MAE: 0.7207264398893781
